# Portfolio 06｜LangGraphを利用した討論エージェント

## 概要

本Notebookでは、ユーザが入力した議題について、2つのDebaterエージェントが異なる立場から議論し、最後にJudge役のLLMが討論内容を評価するエージェントを構築します。

Debater A / Debater B は同じOpenAIモデルを利用し、異なるPersonaを与えることで別のエージェントとして振る舞わせます。Judgeは討論よりも評価の安定性を重視し、別モデルを利用します。

ユーザが入力するのは議題のみです。議論に参加する2人のPersonaは自動生成し、討論終了後に開示します。

また、すべての入力をそのまま討論へ渡すのではなく、最初にTopic Validatorで「議論として成立するか」を判定します。

### 想定する入力例

- 犬派 vs 猫派
- 海派 vs 山派
- 朝型派 vs 夜型派
- 紙の本 vs 電子書籍
- 書籍から学ぶ vs コードを書きながら学ぶ

### 議論対象外とする入力

以下のような入力は議論対象外とします。

- 客観的事実・数値・定義などによって答えが一意に決まるもの
  - 例：日本で一番高い山は？
  - 例：琵琶湖と手賀沼ではどちらが大きいか
- 政治・宗教・戦争・差別など、強い思想的対立を含むテーマ

## 主な構成

1. **Topic Validator**
   - 入力されたテーマが議論可能か判定
2. **Persona Generator**
   - Python側で割り当てた討論アプローチに沿って、Debater A / Debater B の人物像・価値観・思考スタイル・話し方を生成
3. **Debater A / Debater B**
   - 入力順に立場を固定して複数ターン議論
4. **Judge**
   - 議論内容を評価し、判定理由・両者の良かった点を出力

## 使用技術

- OpenAI API
- LangChain
- LangGraph
- LCEL
- Structured Output
- Pydantic
- Role-Based Agent
- Persona
- State / Conditional Edge


## 実行前の注意

- 実行には **OpenAI API Key** が必要です。
- Streamlit版では利用者が画面から自分のAPI Keyを入力し、バックエンドへ引数として渡します。API Keyをソースコードへ保存する構成にはしません。
- OpenAI APIの利用にはAPI利用料金が発生します。
- 利用料金は入力・出力トークン数や使用モデルによって変動します。
- 本Notebookではコストを抑えるため、反復回数の多い処理に軽量モデルを利用し、Judgeのみ別モデルを利用します。
- Debater A → Debater Bを1ターンとして、既定では **2ターン** の討論とします。必要に応じて `MAX_TURNS` を変更できます。

## 設計方針

LangGraphではエージェント全体の状態遷移・ループ・条件分岐を管理し、各Node内部の処理はLCELで構成します。

Debater A / Debater B に能力差は持たせません。違いを持たせるのは、主張を組み立てる **討論アプローチ** と、それに対応する人物像・価値観・話し方です。

### Debater A / Debater B の討論アプローチ

両者には、能力差ではなく次の「アプローチの違い」を与えます。

- **分析型**：条件整理・比較・因果関係・論点の構造化を重視
- **実践型**：具体例・実際の利用場面・実行可能性・体験に基づく説明を重視

どちらか一方が有利にならないよう、知識量・論理性・知能・議論能力については同等として扱います。

分析型だけを「論理的」、実践型を「感覚的」とみなす設計にはしません。実践型も明確な理由と反論を示し、分析型も抽象的な整理だけではなく議題に結びつく根拠を示します。

どちらのアプローチを Debater A / Debater B に割り当てるかは実行ごとにランダムで決定します。

ユーザ入力の立場については入力順を維持します。たとえば「犬派 vs 猫派」なら Debater A は犬派、Debater B は猫派です。ただし、分析型 / 実践型の割り当ては実行ごとに入れ替わります。

Personaは討論開始時にはユーザへ公開せず、Judgeによる最終評価とあわせて表示します。

### 入力の表記揺れ

ユーザ入力では、「犬派vs猫派」「犬派 VS 猫派」「犬と猫ではどちらが良いか」のような表記揺れを許容します。

入力内容はTopic Validatorで解釈し、議論可能な場合は `normalized_topic` として内部で扱いやすい形式へ整形します。


## 1. ライブラリのImportとモデル設定

必要なライブラリを読み込み、Notebook直接実行用のAPI Keyと使用モデルを設定します。

頻繁に呼び出すValidator / Persona Generator / Debaterにはコスト重視モデルを利用し、Judgeのみ評価の安定性を優先したモデルを利用します。

Streamlit版では、同じモデル設定・Prompt・State・Graph構成を `debate_agent.py` に分離し、`run_debate(topic, api_key)` から呼び出します。


In [ ]:
import os
import random

from typing import Literal

from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langgraph.graph import StateGraph, START, END

In [ ]:
# OpenAI API Key の確認
NOTEBOOK_API_KEY = os.getenv("OPENAI_API_KEY")

if not NOTEBOOK_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY が設定されていません。"
        "環境変数にOpenAI API Keyを設定してから再実行してください。"
    )

print("OPENAI_API_KEY: 設定済み")


In [ ]:
# -----------------------------
# 実行設定
# -----------------------------

# 反復回数の多い処理はコスト重視モデルを利用
COST_MODEL = "gpt-5.6-luna"

# Judgeは1回だけ実行するため、評価の安定性を優先して一段上のモデルを利用
JUDGE_MODEL = "gpt-5.6-terra"

# Debater A -> Debater B を1ターンとしてカウント
# コストを抑えるため既定は2ターン。必要に応じて3へ変更可能
MAX_TURNS = 2


# Persona Generator用：人物像にある程度の幅を持たせる
persona_llm = ChatOpenAI(
    model=COST_MODEL,
    temperature=0.7,
    api_key=NOTEBOOK_API_KEY
)

# Validator用：入力テーマの判定を安定させる
validator_llm = ChatOpenAI(
    model=COST_MODEL,
    temperature=0,
    api_key=NOTEBOOK_API_KEY
)

# Debate用：議論にある程度の表現の幅を持たせる
debate_llm = ChatOpenAI(
    model=COST_MODEL,
    temperature=0.7,
    api_key=NOTEBOOK_API_KEY
)

# Judge用：最終評価のみ一段上のモデルを利用
judge_llm = ChatOpenAI(
    model=JUDGE_MODEL,
    temperature=0,
    api_key=NOTEBOOK_API_KEY
)


## 2. Topic Validator

Topic Validatorは、ユーザ入力を `VALID_DEBATE` / `NOT_DEBATABLE` / `RESTRICTED_TOPIC` の3種類に分類します。

判定結果はStructured Outputとして受け取り、議論可能な場合は `normalized_topic` に整形します。LCELでは `Prompt | LLM` の形で処理を接続します。

In [ ]:
# Topic Validator が返す分類結果のスキーマ

class TopicValidationResult(BaseModel):
    category: Literal[
        "VALID_DEBATE",
        "NOT_DEBATABLE",
        "RESTRICTED_TOPIC"
    ] = Field(
        description="入力されたテーマの分類結果"
    )

    reason: str = Field(
        description="その分類にした理由"
    )

    normalized_topic: str | None = Field(
        default=None,
        description="議論可能な場合に、議題として自然な形へ整えた文章"
    )

In [ ]:
# Topic Validator 用プロンプト
validator_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
あなたは討論エージェントの入力テーマを判定するValidatorです。

入力されたテーマを、以下の3種類のいずれかに分類してください。

1. VALID_DEBATE
   - 複数の立場が合理的に成立する
   - 価値観、好み、目的、評価基準によって結論が変わり得る
   - 政治・宗教・戦争・差別など、強い思想的対立を含まない

2. NOT_DEBATABLE
   - 客観的事実、数値、定義などによって答えがほぼ一意に決まる
   - 単なる事実確認や手続き確認である

3. RESTRICTED_TOPIC
   - 政治、宗教、戦争、差別など、今回の討論エージェントでは扱わないテーマ

VALID_DEBATE の場合は、元の意味を変えない範囲で
議論しやすい自然な文章に整えて normalized_topic に格納してください。
対立する2つの立場・選択肢の登場順は変更しないでください。

NOT_DEBATABLE または RESTRICTED_TOPIC の場合は
normalized_topic を null にしてください。

reason は判定根拠が分かる1〜2文程度の簡潔な説明にしてください。
"""
        ),
        (
            "human",
            "入力テーマ: {topic}"
        )
    ]
)

In [ ]:
# Topic Validator のLCEL
validator_chain = (
    validator_prompt
    | validator_llm.with_structured_output(TopicValidationResult)
)

## 3. Persona Generator

Persona Generatorは、議題の入力順を維持したままDebater A / Debater Bの人物像を生成します。

立場は入力順で固定し、**分析型 / 実践型** の討論アプローチだけをPython側でランダムに割り当てます。

- 分析型：条件整理・比較・因果関係・論点の構造化を重視
- 実践型：具体例・利用場面・実行可能性・体験に基づく説明を重視

両者はアプローチが異なるだけで、知識量・論理性・知能・議論能力に優劣は持たせません。


In [ ]:
# Debater 1人分のPersonaスキーマ
class DebaterPersona(BaseModel):
    position: str = Field(
        description="担当する立場"
    )

    style_type: Literal["分析型", "実践型"] = Field(
        description="Debaterに割り当てる討論アプローチの大分類"
    )

    personality: str = Field(
        description="能力差ではなく人物としての違いを表す人物像"
    )

    values: list[str] = Field(
        description="議論で重視する価値観"
    )

    thinking_style: str = Field(
        description="物事をどのように考えるかという思考スタイル"
    )

    speaking_style: str = Field(
        description="議論時の話し方の特徴"
    )


In [ ]:
# Persona Generator が返す結果のスキーマ
class PersonaGenerationResult(BaseModel):
    debater_a: DebaterPersona
    debater_b: DebaterPersona

In [ ]:
# 分析型 / 実践型をA・Bへランダムに割り当てる
# 1回の討論中は固定し、エージェント全体を再実行したときに再割り当てする
PERSONA_STYLE_TYPES = ["分析型", "実践型"]

def assign_persona_styles():
    style_a, style_b = random.sample(PERSONA_STYLE_TYPES, k=2)
    return style_a, style_b


In [ ]:
# Persona Generator 用プロンプト
persona_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
あなたは討論エージェント用のPersona Generatorです。

与えられた議題について、
Debater A と Debater B のPersonaを生成してください。

【ルール】

- 議題に登場する順番を維持してください。
- 最初の立場を Debater A、次の立場を Debater B としてください。
- Debater A の style_type は「{style_a}」、Debater B の style_type は「{style_b}」にしてください。

【討論アプローチ】

- 分析型は、条件整理・比較・因果関係・論点の構造化を重視してください。
- 実践型は、具体例・実際の利用場面・実行可能性・体験に基づく説明を重視してください。
- 分析型と実践型はアプローチが異なるだけで、知識量・論理性・知能・議論能力には優劣をつけないでください。
- 実践型を感覚的・非論理的な人物として生成しないでください。明確な理由を示し、相手の論点へ具体的に応答できる人物にしてください。
- 分析型を自動的に高度・優秀な人物として生成しないでください。整理した論点を必ず議題や具体的な根拠へ結びつける人物にしてください。

【Persona】

- 違いを持たせるのは人物像、価値観、思考スタイル、話し方です。
- 一方だけが極端、非合理的、攻撃的にならないようにしてください。
- どちらの立場も合理的に主張できる人物にしてください。
- 年齢、性別など議論に不要な属性は設定しないでください。
- 2人のPersonaは明確に異なるものにしてください。
- Personaの違いが議論内容や話し方に反映されるようにしてください。
- 親しみやすく自然な話し方の特徴を設定してください。
- 攻撃的、断定的すぎる話し方にはしないでください。
- personality / thinking_style / speaking_style はそれぞれ1〜2文程度にしてください。
- values は各Debaterにつき3項目程度にしてください。
"""
        ),
        (
            "human",
            "議題: {topic}"
        )
    ]
)


In [ ]:
# Persona Generator のLCEL
persona_chain = (
    persona_prompt
    | persona_llm.with_structured_output(PersonaGenerationResult)
)

## 4. Debater

Debater A / Debater B は共通のLCELチェーンを利用し、Stateに保存されたPersonaだけを差し替えて発言します。

討論履歴はStateへすべて保存しますが、次の発言生成時にLLMへ渡すのは **相手の直前の発言のみ** とします。これにより、ターン数の増加に伴う入力トークンの膨張を抑えます。Judgeだけは最後に議論履歴全体を参照します。

`style_type` もDebaterへ渡し、分析型 / 実践型の違いが発言に反映されるようにします。ただし、どちらのタイプでも明確な理由・具体性・相手への応答を求め、能力差にはしません。


In [ ]:
# Debater 用プロンプト
debater_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
あなたは討論に参加するDebaterです。

以下のPersonaと立場を維持しながら議論してください。

【基本ルール】

- 自分の立場を一貫して支持してください。
- Personaの人物像、価値観、思考スタイル、話し方を反映してください。
- 初回発言では、自分の立場の主張から始めてください。
- 相手の発言がある場合は、その内容を一度受け止めたうえで自分の立場から反論してください。
- 相手を攻撃したり、人格を否定したりしないでください。
- 親しみやすく自然な口調で話してください。
- 単なる感想だけではなく、明確な理由や具体例を含めてください。
- 1回の発言は250〜400文字程度を目安にしてください。
- 同じ主張の繰り返しを避け、相手の直前の発言に対して新しい論点を1つ以上加えてください。

【討論アプローチ】

あなたの討論アプローチは「{style_type}」です。

- 分析型の場合は、条件整理・比較・因果関係・論点の構造化を中心に説得してください。
- 実践型の場合は、具体例・実際の利用場面・実行可能性・体験に基づく説明を中心に説得してください。
- どちらのアプローチでも、主張と理由のつながりを明確にし、相手の論点へ具体的に応答してください。
- 分析型だから論理性が高い、実践型だから感覚的という扱いにはしないでください。

【あなたの立場】
{position}

【あなたの人物像】
{personality}

【重視する価値観】
{values}

【思考スタイル】
{thinking_style}

【話し方】
{speaking_style}
"""
        ),
        (
            "human",
            """
【議題】
{topic}

【相手の直前の発言】
{opponent_message}

あなたの次の発言を生成してください。
"""
        )
    ]
)


In [ ]:
# Debater のLCEL
debater_chain = (
    debater_prompt
    | debate_llm
    | StrOutputParser()
)

## 5. StateとNode

`DebateState` はLangGraph全体で共有する状態です。

ユーザ入力、正規化された議題、Persona、討論履歴、ターン数、Validatorの判定、Judge結果を保持します。各NodeはStateから必要な値を取り出し、更新する項目だけを返します。

In [ ]:
# Debate Agent 全体で共有するState
class DebateState(BaseModel):
    topic: str = Field(
        description="ユーザが入力した元の議題"
    )

    normalized_topic: str = Field(
        default="",
        description="Topic Validator が整形した議題"
    )

    persona_a: DebaterPersona | None = Field(
        default=None,
        description="Debater A のPersona"
    )

    persona_b: DebaterPersona | None = Field(
        default=None,
        description="Debater B のPersona"
    )

    debate_history: list[str] = Field(
        default_factory=list,
        description="Debater A / B の発言履歴"
    )

    turn_count: int = Field(
        default=0,
        description="現在の議論ターン数"
    )

    validation_category: str = Field(
        default="",
        description="Topic Validator の分類結果"
    )

    validation_reason: str = Field(
        default="",
        description="Topic Validator の判定理由"
    )

    judgment: dict = Field(
        default_factory=dict,
        description="Judge の最終判定結果"
    )

In [ ]:
# Topic Validator Node
def topic_validator_node(state: DebateState):
    result = validator_chain.invoke(
        {"topic": state.topic}
    )

    return {
        "normalized_topic": result.normalized_topic or "",
        "validation_category": result.category,
        "validation_reason": result.reason
    }

In [ ]:
# Persona Generator Node
def persona_generator_node(state: DebateState):
    # 実行ごとに分析型 / 実践型をランダムに割り当てる
    style_a, style_b = assign_persona_styles()

    result = persona_chain.invoke(
        {
            "topic": state.normalized_topic,
            "style_a": style_a,
            "style_b": style_b
        }
    )

    return {
        "persona_a": result.debater_a,
        "persona_b": result.debater_b
    }


In [ ]:
# Debater A Node
def debater_a_node(state: DebateState):
    persona = state.persona_a

    # 初回以外では、直前のDebater Bの発言だけを渡す
    opponent_message = (
        state.debate_history[-1]
        if state.debate_history
        else "まだ相手の発言はありません。"
    )

    response = debater_chain.invoke(
        {
            "topic": state.normalized_topic,
            "position": persona.position,
            "style_type": persona.style_type,
            "personality": persona.personality,
            "values": "、".join(persona.values),
            "thinking_style": persona.thinking_style,
            "speaking_style": persona.speaking_style,
            "opponent_message": opponent_message
        }
    )

    return {
        "debate_history": state.debate_history + [
            f"Debater A:\n{response}"
        ]
    }


In [ ]:
# Debater B Node
def debater_b_node(state: DebateState):
    persona = state.persona_b

    # 直前のDebater Aの発言だけを渡す
    opponent_message = state.debate_history[-1]

    response = debater_chain.invoke(
        {
            "topic": state.normalized_topic,
            "position": persona.position,
            "style_type": persona.style_type,
            "personality": persona.personality,
            "values": "、".join(persona.values),
            "thinking_style": persona.thinking_style,
            "speaking_style": persona.speaking_style,
            "opponent_message": opponent_message
        }
    )

    return {
        "debate_history": state.debate_history + [
            f"Debater B:\n{response}"
        ],
        "turn_count": state.turn_count + 1
    }


In [ ]:
# Debater B の発言完了を1ターンとしてカウントし、継続またはJudgeへ分岐
def route_after_debate(state: DebateState):
    if state.turn_count < MAX_TURNS:
        return "debater_a"
    return "judge"

## 6. Judge

Judgeは討論終了後に1回だけ実行し、議論履歴全体を参照して評価します。

評価対象はテーマ自体の正解ではなく、**今回の討論でどちらがより説得力のある議論を行ったか**です。

分析型 / 実践型のどちらかが構造的に有利にならないよう、話し方の形式そのものではなく、議題に対してどれだけ有効な根拠・反論・具体性を示せたかを評価します。

特に、分析型の「整理されていること」自体を加点せず、実践型の具体例・利用場面・実行可能性も、主張を支える根拠として同等に評価するようPromptで指定します。

判定結果は自由な文字列にせず、`Literal` を使って `A` / `B` / `DRAW` の3種類に限定します。


In [ ]:
# Judge が返す評価結果のスキーマ
class JudgeResult(BaseModel):
    # Literalは指定した候補のいずれかだけを許可する型
    winner: Literal[
        "A",
        "B",
        "DRAW"
    ] = Field(
        description="討論の判定結果。Aの主張がより説得的ならA、BならB、実質的な差がなければDRAW"
    )

    reason: str = Field(
        description="判定理由。評価基準に基づき簡潔に説明する"
    )

    strengths_a: str = Field(
        description="Debater A の良かった点"
    )

    strengths_b: str = Field(
        description="Debater B の良かった点"
    )


In [ ]:
# Judge 用プロンプト
judge_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
あなたは討論を公平に評価するJudgeです。

Debater A と Debater B の議論全体を読み、
「今回の討論でどちらがより説得力のある議論を行ったか」を評価してください。

【評価基準】

- 議題に直接答えているか
- 主張と理由のつながりが明確か
- 相手の主張を理解したうえで具体的に反論できているか
- 根拠・具体例・利用場面が主張を十分に支えているか
- 反論を受けても、自分の立場を一貫して発展させられているか

【スタイルの公平性】

- Debaterの討論アプローチや話し方の好みを判定材料にしないでください。
- 分析型の「整理されている」「比較項目が多い」「文章が構造化されている」といった形式そのものを加点しないでください。
- 実践型の具体例・実際の利用場面・実行可能性・体験に基づく説明も、主張を適切に支えている場合は分析的な整理と同等に評価してください。
- 実践型を感覚的・非論理的とみなしたり、分析型をより知的・高度とみなしたりしないでください。
- 重要なのは発言の形式ではなく、議題に対してどれだけ有効な根拠と反論を提示できたかです。

【判定ルール】

- Debaterの立場そのものを理由に有利・不利をつけないでください。
- 外部の情報を持ち込まず、実際に行われた議論内容のみを評価してください。
- テーマ自体が主観的でも、討論の質に差があれば A または B を選んでください。
- DRAW は、上記の評価基準を比較しても実質的な差が見つからない場合だけ選んでください。
- reason は2〜4文程度、strengths_a / strengths_b は各1〜2文程度で簡潔にしてください。
"""
        ),
        (
            "human",
            """
【議題】
{topic}

【討論内容】
{debate_history}

討論全体を評価してください。
"""
        )
    ]
)


In [ ]:
# Judge のLCEL
judge_chain = (
    judge_prompt
    | judge_llm.with_structured_output(JudgeResult)
)

In [ ]:
# Judge Node
def judge_node(state: DebateState):
    history_text = "\n\n".join(state.debate_history)

    result = judge_chain.invoke(
        {
            "topic": state.normalized_topic,
            "debate_history": history_text
        }
    )

    return {
        "judgment": result.model_dump()
    }

## 7. 入力判定の分岐

Topic Validatorの結果によって次のNodeを切り替えます。

- `VALID_DEBATE` → Persona Generatorへ進む
- `NOT_DEBATABLE` → 討論せず終了
- `RESTRICTED_TOPIC` → 討論せず終了

表示処理はGraph内部では行わず、NotebookまたはStreamlit側で `validation_category` と `validation_reason` を参照してユーザへ案内します。

これにより、エージェント本体とUI表示を分離します。


In [ ]:
# Topic Validator の判定結果によって次のNodeを決定
def route_after_validation(state: DebateState):
    if state.validation_category == "VALID_DEBATE":
        return "persona_generator"

    elif state.validation_category == "NOT_DEBATABLE":
        return "not_debatable"

    return "restricted_topic"

In [ ]:
# 議論として成立しない場合
# 表示はNotebook / Streamlit側で行うため、Graph内部ではStateを変更しない
def not_debatable_node(state: DebateState):
    return {}


In [ ]:
# 今回のエージェントでは扱わないテーマの場合
# 表示はNotebook / Streamlit側で行うため、Graph内部ではStateを変更しない
def restricted_topic_node(state: DebateState):
    return {}


## 8. LangGraphの構築

ここまでに作成したNodeをStateGraphへ登録し、Edgeで接続します。

```text
START
  ↓
Topic Validator
  ├─ NOT_DEBATABLE ─────→ END
  ├─ RESTRICTED_TOPIC ──→ END
  └─ VALID_DEBATE
          ↓
    Persona Generator
          ↓
      Debater A
          ↓
      Debater B
          ↓
      Turn Check
       ├─ 継続 → Debater A
       └─ 終了 → Judge → END
```

In [ ]:
# Debate Agent のGraphを作成
graph = StateGraph(DebateState)

# Nodeを登録
graph.add_node("topic_validator", topic_validator_node)
graph.add_node("persona_generator", persona_generator_node)
graph.add_node("debater_a", debater_a_node)
graph.add_node("debater_b", debater_b_node)
graph.add_node("judge", judge_node)

graph.add_node("not_debatable", not_debatable_node)
graph.add_node("restricted_topic", restricted_topic_node)

In [ ]:
# 開始地点
graph.add_edge(START, "topic_validator")

In [ ]:
# Topic Validator 後の条件分岐
graph.add_conditional_edges(
    "topic_validator",
    route_after_validation,
    {
        "persona_generator": "persona_generator",
        "not_debatable": "not_debatable",
        "restricted_topic": "restricted_topic"
    }
)

In [ ]:
# Persona生成後、Debater Aから議論開始
graph.add_edge("persona_generator", "debater_a")

# Aの発言後はBへ
graph.add_edge("debater_a", "debater_b")

In [ ]:
# 規定ターン数まで議論を続ける
graph.add_conditional_edges(
    "debater_b",
    route_after_debate,
    {
        "debater_a": "debater_a",
        "judge": "judge"
    }
)

In [ ]:
# Judge終了後
graph.add_edge("judge", END)

# 議論対象外の場合
graph.add_edge("not_debatable", END)
graph.add_edge("restricted_topic", END)

In [ ]:
# Graphを実行可能な形にコンパイル
debate_agent = graph.compile()

## 9. ユーザ向け表示と実行

最後にNotebookで直接実行する場合の表示を整えます。

- 起動時にAPI Key・課金・入力例・議論対象外テーマを案内
- 討論中はPersonaを非公開
- 討論終了後に議論履歴を表示
- 最後にPersona、Judgeの判定・理由・両者の良かった点を表示
- `NOT_DEBATABLE` / `RESTRICTED_TOPIC` の案内はGraph内部ではなく、この表示層で行う

Streamlit版でも同じ考え方で、`debate_agent.py` は判定結果を返し、`app.py` が表示を担当します。


In [ ]:
# 起動時の案内
def show_usage():
    print("=" * 60)
    print("Debate Agent")
    print("=" * 60)
    print("複数の立場が合理的に成立する、日常的で比較可能なテーマを入力してください。")
    print()
    print("【必要なもの】")
    print("- OpenAI API Key")
    print("- OpenAI APIの利用にはAPI利用料金が発生します")
    print("- 利用料金は入力内容・出力内容・モデル料金によって変動します")
    print()
    print("【議論できる例】")
    print("- 犬派 vs 猫派")
    print("- 海派 vs 山派")
    print("- 紙の本 vs 電子書籍")
    print("- 書籍から学ぶ vs コードを書きながら学ぶ")
    print()
    print("【入力について】")
    print("- 『犬派vs猫派』『犬派 VS 猫派』『犬と猫ではどちらが良いか』など、")
    print("  表記揺れがあっても入力できます。")
    print()
    print("【討論アプローチ】")
    print("- 分析型：条件整理・比較・因果関係・論点の構造化を重視")
    print("- 実践型：具体例・利用場面・実行可能性・体験に基づく説明を重視")
    print("- 両者に知識量・論理性・知能・議論能力の差は設けません")
    print()
    print("【議論しないテーマ】")
    print("- 客観的事実によって答えが一意に決まる質問")
    print("- 政治・宗教・戦争・差別など、強い思想的対立を含むテーマ")
    print("=" * 60)


def show_debate_history(result):
    print()
    print("=" * 60)
    print("DEBATE")
    print(f"議題: {result['normalized_topic']}")
    print("=" * 60)

    for message in result["debate_history"]:
        print(message)
        print("-" * 60)


def show_final_result(result):
    judgment = result["judgment"]
    winner = judgment["winner"]

    persona_a = result["persona_a"]
    persona_b = result["persona_b"]

    print()
    print("=" * 60)
    print("今回のPersona")
    print("分析型 / 実践型は能力差ではなく、主張を組み立てるアプローチの違いです。")
    print("=" * 60)

    print(
        f"Debater A\n"
        f"討論アプローチ: {persona_a.style_type}\n"
        f"立場: {persona_a.position}\n"
        f"人物像: {persona_a.personality}"
    )

    print()

    print(
        f"Debater B\n"
        f"討論アプローチ: {persona_b.style_type}\n"
        f"立場: {persona_b.position}\n"
        f"人物像: {persona_b.personality}"
    )

    print()
    print("=" * 60)

    if winner == "A":
        print("判定: Debater A の主張がより説得的")
        print(f"主張: {persona_a.position}")

    elif winner == "B":
        print("判定: Debater B の主張がより説得的")
        print(f"主張: {persona_b.position}")

    else:
        print("判定: 両者の説得力は同程度")

    print("=" * 60)

    print(f"\n判定理由:\n{judgment['reason']}")

    print(f"\nDebater A の良かった点:\n{judgment['strengths_a']}")

    print(f"\nDebater B の良かった点:\n{judgment['strengths_b']}")


In [ ]:
# ユーザ入力の前に利用方法を表示
show_usage()


In [ ]:
# ユーザから議題を受け取り、Debate Agentを実行
topic = input("議題を入力してください：").strip()

if not topic:
    print("議題が入力されていません。")
else:
    initial_state = DebateState(topic=topic)
    result = debate_agent.invoke(initial_state)

    category = result["validation_category"]

    if category == "VALID_DEBATE":
        show_debate_history(result)
        show_final_result(result)

    elif category == "NOT_DEBATABLE":
        print("この入力は、客観的な事実などによって答えが決まるため、")
        print("討論テーマとしては扱いません。")
        print()
        print(f"理由: {result['validation_reason']}")

    elif category == "RESTRICTED_TOPIC":
        print("このエージェントでは、政治・宗教・戦争・差別など、")
        print("強い思想的対立を含むテーマは討論対象としていません。")
        print()
        print(f"理由: {result['validation_reason']}")


## 10. Streamlit版との役割分離

本Notebookは、LangGraph / LCEL / Structured Output / Persona / Judge の構成を段階的に確認するための技術解説版です。

Web UI版では同じPrompt・Schema・State・Node・Graph構成を `debate_agent.py` に分離し、Streamlitの `app.py` から次のインターフェースで呼び出します。

```python
result = run_debate(topic, api_key)
```

役割は次のように分離します。

```text
app.py
  ├─ OpenAI API Key の入力
  ├─ 議題の入力
  └─ 結果の表示
        ↓
debate_agent.py
  ├─ LLM / LCEL
  ├─ Topic Validator
  ├─ Persona Generator
  ├─ Debater A / B
  ├─ Judge
  └─ LangGraph
```

`debate_agent.py` はimport時にAPI Keyを要求せず、`run_debate(topic, api_key)` の実行時に利用者のKeyを明示的に受け取ります。

これにより、GitHubへAPI Keyを含めず、別環境でも利用者自身のOpenAI API Keyで実行できる構成にします。
